# AI Code Trust Pipeline — APPROVED / REJECTED / NEEDS HUMAN REVIEW

**Question:** given a piece of AI-generated code, can a model decide whether to trust it
*before* anyone runs the hidden test suite?

**Design principles**
- **Labels never come from AI judgment.** A candidate is *trustworthy* when it passes MBPP's
  **hidden** tests and has no medium/high-severity security finding (Bandit).
- **Features only use information available at decision time:** the code itself, static
  analysis, the **one visible test** shown in the prompt, agreement between several samples for
  the same problem (self-consistency), and the generating model's token confidence. Hidden-test
  results are **never** features.
- **Humans validate the labels; they don't produce training data.** A stratified sample is reviewed
  (real, or simulated with `HUMAN_ADDED_FEEDBACK = True` for pipeline demos) and agreement with the
  automated label is reported via Cohen's kappa.
- **No leakage across problems:** train / calibration / test splits are grouped by `task_id`.
- **Three-way output:** a calibrated probability becomes APPROVED / REJECTED / NEEDS HUMAN REVIEW
  using thresholds tuned on a calibration split for a target precision, with split-conformal
  prediction as a distribution-free alternative.

**Code layout.** All logic lives in the `trust_pipeline/` package next to this notebook; the same
stages run from the command line with `uv run trust-pipeline`. This notebook calls the
stages one at a time and shows what each produces.

> Start with `SMOKE_TEST = True` (no model needed, a few minutes on CPU) to check everything works,
> then switch to real generation with a small `NUM_PROBLEMS` before scaling up.

## 1. Install dependencies and import the package

**Locally with uv (recommended).** From the project folder:

```bash
uv sync --extra notebook                 # core + Jupyter
uv sync --extra notebook --extra llm     # + local Hugging Face generation (torch, transformers)
uv sync --extra notebook --extra api     # + OpenAI / Anthropic backends
uv run jupyter lab                       # open this notebook with the project's environment
# or register a kernel for VS Code / Cursor and select "mle-trust":
uv run python -m ipykernel install --user --name mle-trust
```

Then skip the next cell.

**Colab (no uv environment).** Upload or `git clone` the project folder, `%cd` into it, and run the
next cell. It installs uv and uses it to install this project into Colab's Python. Colab's
preinstalled torch satisfies the `llm` extra, so it isn't reinstalled.

In [ ]:
# Colab only. Locally, use `uv sync --extra notebook` and skip this cell.
import sys
if "google.colab" in sys.modules:
    !pip install -q uv
    !uv pip install --system -q -e ".[llm,api]"

In [ ]:
import os
import sys

try:
    import trust_pipeline  # noqa: F401  (installed by `uv sync` / the Colab cell above)
except ImportError:
    sys.path.insert(0, os.path.abspath("."))
    try:
        import trust_pipeline  # noqa: F401
    except ImportError as e:
        raise ImportError("trust_pipeline not found: run `uv sync --extra notebook` in the project folder "
                          "and start Jupyter with `uv run jupyter lab` (Colab: run the cell above).") from e

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from trust_pipeline import human_review, plots
from trust_pipeline.config import Config
from trust_pipeline.execution import Executor
from trust_pipeline.generation import build_prompt
from trust_pipeline.astutils import make_mutants
from trust_pipeline.pipeline import (ArtifactPaths, stage_features, stage_generate, stage_human,
                                     stage_label, stage_prepare, stage_train)

## 2. Configuration

Use **several model/setting combinations with sampling (`temperature > 0`)** so the dataset has a
realistic mix of correct and incorrect code, and so every problem has `SAMPLES_PER_CONFIG` samples
for the self-consistency features. The default local models are small *instruction-tuned* code
models (`Qwen2.5-Coder` 0.5B and 1.5B), which follow "return only a function" prompts much better
than base models. See `trust_pipeline/config.py` for the model list and every other setting.

In [ ]:
from getpass import getpass

SMOKE_TEST = False            # True: reference solutions + AST mutants, no LLM (pipeline check only)
NUM_PROBLEMS = 100            # None -> all 974 MBPP problems
SAMPLES_PER_CONFIG = 5
HUMAN_ADDED_FEEDBACK = True   # True: simulated reviewer (demo only); False: blank CSV for real review
USE_OPENAI = False
USE_ANTHROPIC = False

cfg = Config(
    smoke_test=SMOKE_TEST,
    num_problems=NUM_PROBLEMS,
    samples_per_config=SAMPLES_PER_CONFIG,
    human_added_feedback=HUMAN_ADDED_FEEDBACK,
    use_local_hf=True,
    use_openai=USE_OPENAI,
    use_anthropic=USE_ANTHROPIC,
    use_mutant_augmentation=False,     # mutated references as training-only rows
    target_approve_precision=0.95,
    target_reject_precision=0.95,
    conformal_alpha=0.10,
    output_dir="artifacts_smoke" if SMOKE_TEST else "artifacts",
)

if not SMOKE_TEST:
    if USE_OPENAI and not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
    if USE_ANTHROPIC and not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

paths = ArtifactPaths(cfg.output_dir)
executor = Executor.from_config(cfg)
print(f"SMOKE_TEST={cfg.smoke_test}; model configs: {cfg.model_names}; artifacts -> {cfg.output_dir}/")

## 3. Sandboxed execution harness

Every candidate runs in its own Python process (`trust_pipeline/execution.py`). Each assertion and
probe gets its own time limit, so one slow probe can't turn a correct solution into a `timeout`
label. The process also has a memory cap, a scratch working directory and a fixed
`PYTHONHASHSEED`, so `repr()` of sets can be compared across candidates.

Each candidate is run twice:
- **feature run:** module definitions, the visible test and probe inputs (all known at decision time)
- **label run:** the hidden tests only (ground truth)

In [ ]:
print(executor.execute("def add(a, b):\n    return a + b", tests=["assert add(1, 2) == 3"], probes=["add(2, 2)"]))
print(executor.execute("def add(a, b):\n    while True: pass", tests=["assert add(1, 2) == 3"]))

## 4. Load MBPP and split tests into visible vs. hidden

All four splits of the `full` config are pooled (974 problems) and a random subset of
`NUM_PROBLEMS` is taken. For each problem:
- **visible test** = `test_list[0]`: shown in the prompt and used for features
- **hidden tests** = `test_list[1:]` + `challenge_test_list`: used **only** for the label
- **probes** = the visible call with perturbed arguments, used for self-consistency

Reference solutions are run against all tests first, and problems whose reference fails are
dropped. The reference solution is never shown to the generating model. If the Hugging Face Hub
can't be reached, the loader falls back to the local cache.

In [ ]:
problems = stage_prepare(cfg, paths, executor)
print()
print(build_prompt(problems[0]))
problems[0]

## 5. Generate candidates

LLM generations are cached in `artifacts/generations.jsonl`, so an interrupted run resumes where
it stopped. Failed API calls are not cached. Each sample stores the generating model's token
log-probabilities (mean and minimum); Anthropic doesn't expose them, so they're imputed.

Every row records its **provenance** (`llm`, `reference` or `mutant`). Mutants are only used as
training data, so the evaluation reflects naturally generated code. In `SMOKE_TEST` mode the
candidates are each problem's reference solution plus single-bug mutants (flipped comparison,
swapped operator, off-by-one, `and`/`or` swap, dropped return value, misspelled name).

In [ ]:
candidates = stage_generate(cfg, paths, problems)
candidates.head()

## 6. Execute candidates, static analysis and labels

**Trustworthy (1)** = passes every hidden test **and** has no medium/high Bandit finding.
`exec_status` keeps the detailed outcome (`pass`, `wrong_output`, `runtime_error`, `timeout`,
`syntax_error`).

"NEEDS HUMAN REVIEW" is deliberately *not* a ground-truth class, because running the tests can't
produce it. It is a decision the system makes when the model's confidence is too low (section 11).

In [ ]:
labeled = stage_label(cfg, paths, problems, candidates, executor)
labeled[["candidate_id", "exec_status", "hidden_pass_frac", "visible_pass", "ruff_issues",
         "bandit_med_high", "label_trustworthy"]].head()

## 7. Features

- **Self-consistency.** Each candidate's outputs on the probe inputs form a *behaviour signature*.
  `behavior_agreement` is the share of sibling candidates with an identical signature;
  `behavior_agreement_visible` counts only siblings that pass the visible test. Mutants are scored
  against the pool but never added to it.
- **Code structure.** AST counts, recursion, whether the expected function is defined, and radon
  complexity / maintainability.
- **Static analysis, visible test, generation confidence, model config.**

The feature list is **explicit**, and a guard raises an error if anything derived from the hidden
tests or the label slips in. Features with no information in a run (for example log-probabilities
in `SMOKE_TEST`) are dropped.

In [ ]:
dataset, FEATURE_COLUMNS = stage_features(cfg, paths, problems, labeled)
dataset[["behavior_agreement", "behavior_agreement_visible", "probe_error_frac"]].describe()

## 8. Human validation sample (real or simulated)

The human review answers one question: **does the automated label match what a careful reviewer
would call trustworthy?** It never produces training labels.

- **`HUMAN_ADDED_FEEDBACK = True`:** a *simulated* reviewer labels the sample so the agreement
  analysis runs end-to-end. It mostly agrees with the automated label, but sometimes rejects messy
  passing code, sometimes misses subtle bugs, and adds noise. **Don't report these numbers as
  findings.**
- **`HUMAN_ADDED_FEEDBACK = False`:** writes `human_review_sample.csv` with a blank
  `human_trustworthy` column. Reviewers read the problem and code (hide `exec_status` and
  `label_trustworthy` from them) and enter `yes` / `no`. Re-running this stage, or
  `uv run trust-review`, computes Cohen's kappa without overwriting the reviews.

In [ ]:
agreement = stage_human(cfg, paths, dataset)
agreement

## 9. Exploratory analysis

In [ ]:
plots.plot_eda(dataset)
plt.show()

print("How much does the visible test alone tell us?")
display(pd.crosstab(dataset["visible_pass"], dataset["label_trustworthy"],
                    rownames=["visible_pass"], colnames=["trustworthy"]))
passing = dataset[dataset["exec_status"] == "pass"]
if len(passing):
    print(f"Of candidates passing the hidden tests: {(passing.ruff_issues > 0).mean():.1%} have lint "
          f"findings, {(passing.bandit_issues > 0).mean():.1%} have security findings.")

## 10. Train and evaluate

1. **Grouped split** by `task_id`: train 60%, calibration 20%, test 20%.
2. **Baselines:** majority class, and logistic regression on `visible_pass` alone. If the full model
   can't beat the visible-test baseline, the extra features aren't helping.
3. **Candidates:** logistic regression, random forest, histogram gradient boosting, compared with
   grouped 5-fold CV ROC AUC on the training split.
4. The best model is **calibrated** (sigmoid, or isotonic with ≥1,000 rows) using the same folds.
5. **Test-set metrics:** accuracy, precision, recall, F1, ROC AUC, PR AUC and Brier score.

Artifacts (model `.joblib`, `metrics.json`, CSVs and figures) are written to the artifacts folder.

In [ ]:
results = stage_train(cfg, paths, dataset, FEATURE_COLUMNS)
display(results["split_table"])
display(results["cv_table"].round(3))
display(results["comparison"].round(3))

### Confusion matrix, ROC curves and calibration

In [ ]:
plots.plot_evaluation(results["y_test"], results["test_proba"], results["calibrated_name"])
plt.show()

## 11. Three-way trust decision: APPROVED / REJECTED / NEEDS HUMAN REVIEW

**Threshold rule.** On the calibration split, pick
- `t_high`: the lowest threshold (at or above 0.5) where the approved set reaches
  `target_approve_precision`
- `t_low`: the highest threshold (below 0.5) where the rejected set is at least
  `target_reject_precision` untrustworthy

APPROVED if `p ≥ t_high`, REJECTED if `p ≤ t_low`, otherwise NEEDS HUMAN REVIEW. If a target can't
be met, that decision is disabled (everything goes to review) rather than the target being relaxed.

**Split-conformal prediction.** With nonconformity score `1 − p(true class)` on the calibration
split, each candidate gets a prediction set that contains the true label with probability
≥ `1 − alpha`. A set of `{trustworthy}` means APPROVED, `{untrustworthy}` means REJECTED, and
both means NEEDS HUMAN REVIEW.

With a small calibration split the thresholds are noisy; more problems tighten them.

In [ ]:
m = results["metrics"]
print(f"Threshold rule: REJECT if p <= {m['thresholds']['t_low']:.3f}, APPROVE if p >= {m['thresholds']['t_high']:.3f}")
print(f"Conformal: qhat = {m['conformal']['qhat']:.3f}, empirical test coverage = "
      f"{m['conformal']['test_coverage']:.3f} (target >= {1 - cfg.conformal_alpha:.2f})")
display(results["decisions"].round(3))
display(pd.crosstab(pd.Series(results["threshold_decisions"], name="decision"),
                    pd.Series(results["y_test"].values, name="actually trustworthy")))

### Risk–coverage curve

Sort test candidates by confidence `max(p, 1 − p)` and automate the most confident fraction. The
curve shows the error rate among automated decisions at every level of automation, which is the
key trade-off for an agentic trust system. The two operating points are marked.

In [ ]:
plots.plot_risk_coverage(results["y_test"], results["p_test"],
                         {"threshold rule": results["threshold_decisions"],
                          "conformal": results["conformal_decisions"]})
plt.show()

### Example decisions from the test set

In [ ]:
preds = results["predictions"]
display(pd.concat([g.sample(min(3, len(g)), random_state=cfg.seed) for _, g in preds.groupby("decision")])
        [["task_id", "model_name", "p_trustworthy", "decision", "conformal_decision", "exec_status",
          "visible_pass", "behavior_agreement", "generated_code"]])

## 12. Which features drive the prediction?

Permutation importance on the test set: the drop in ROC AUC when a feature is shuffled.

In [ ]:
if len(results["importances"]):
    plots.plot_feature_importance(results["importances"])
    plt.show()

## 13. Scoring a new candidate

`TrustRecommender.recommend` runs the same decision-time pipeline on new code: the visible test,
probes, static analysis and code features. Pass sibling samples for the same problem to get
self-consistency features. The saved model can be used the same way from the command line:
`uv run trust-recommend --task-id 17 --code-file solution.py --sibling other.py`.

In [ ]:
recommender = results["recommender"]
demo = next(p for p in problems if p["task_id"] == results["predictions"]["task_id"].iloc[0])
mutants = [m for m, _ in make_mutants(demo["reference_code"], 3, seed=0)]
print("Problem:", demo["text"])
print("Reference solution ->", recommender.recommend(demo, demo["reference_code"], mutants, executor))
if mutants:
    print("Mutated solution   ->", recommender.recommend(demo, mutants[0],
                                                         [demo["reference_code"]] + mutants[1:], executor))

## 14. Human vs. automated agreement (Cohen's kappa)

This step shows whether the automated label means what we claim. With
`HUMAN_ADDED_FEEDBACK = True` the "human" labels are simulated, so the number only checks the
mechanics. With real reviews, κ > 0.6 is usually read as substantial agreement. Look at the
disagreements to refine the label definition, for example whether lint-heavy passing code should
count as trustworthy. Re-run this cell after filling in the review CSV.

In [ ]:
agreement = human_review.human_agreement(paths.human_review, cfg.human_added_feedback)
if agreement is None:
    print(f"No human labels yet: fill in 'human_trustworthy' in {paths.human_review} and re-run.")
else:
    print(f"Reviewed rows: {agreement['reviewed_rows']}")
    print(f"Raw agreement: {agreement['raw_agreement']:.1%}")
    print(f"Cohen's kappa: {agreement['cohen_kappa']:.3f}")
    print(agreement["confusion"])
    if agreement["simulated"]:
        print("These 'human' labels are SIMULATED. Replace them with real reviews before reporting this number.")

## 15. Run report for your write-up

Every stage records its numbers in `run_report.json`. The training stage then renders
`run_report.md` (tables ready to paste into a report) and adds one summary row per run to
`runs_history.csv`, which sits next to the output folder, so pilot / medium / full runs can be compared side by side.
The report covers run settings and versions, stage timings, generation stats per model config,
execution outcomes, pass@k, how often a visible-test pass still fails hidden tests, splits, CV (mean ± std),
test metrics with 95% bootstrap CIs (resampled by problem), three-way decisions and human agreement.

In [ ]:
from IPython.display import Markdown

with open(paths.run_report_md) as f:
    display(Markdown(f.read()))

## 16. Optional: frozen pretrained code encoder (towards Idea 2)

Set `USE_CODE_EMBEDDINGS = True` to embed *problem + code* with a pretrained code encoder
(UniXcoder by default; mean-pooled, frozen), reduce it to 32 dimensions with PCA fitted on the
training split, append it to the engineered features, and compare on the **same** grouped test
split. Fine-tuning the encoder end-to-end with a classification head is the natural next step.

Requires the `llm` extra (`uv sync --extra notebook --extra llm`).

In [ ]:
USE_CODE_EMBEDDINGS = False
EMBEDDING_MODEL = "microsoft/unixcoder-base"

if USE_CODE_EMBEDDINGS:
    # Needs the llm extra: `uv sync --extra notebook --extra llm`
    import torch
    from sklearn.decomposition import PCA
    from sklearn.ensemble import HistGradientBoostingClassifier
    from tqdm.auto import tqdm
    from transformers import AutoModel, AutoTokenizer
    from trust_pipeline.training import binary_metrics

    device = "cuda" if torch.cuda.is_available() else "cpu"
    enc_tok = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
    encoder = AutoModel.from_pretrained(EMBEDDING_MODEL).to(device).eval()

    def embed(texts, batch_size=32):
        out = []
        for i in tqdm(range(0, len(texts), batch_size), desc="embedding"):
            batch = enc_tok(texts[i:i + batch_size], padding=True, truncation=True,
                            max_length=512, return_tensors="pt").to(device)
            with torch.no_grad():
                hidden = encoder(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1).float()
            out.append(((hidden * mask).sum(1) / mask.sum(1)).cpu().numpy())
        return np.vstack(out)

    train_df, test_df = results["splits"]["train"], results["splits"]["test"]
    texts = lambda f: (f["problem_text"] + "\n" + f["generated_code"]).tolist()
    E_train, E_test = embed(texts(train_df)), embed(texts(test_df))
    pca = PCA(n_components=min(32, E_train.shape[1], len(E_train) - 1), random_state=cfg.seed).fit(E_train)
    emb_cols = [f"emb_{i}" for i in range(pca.n_components_)]
    Xe_train = pd.concat([train_df[FEATURE_COLUMNS].reset_index(drop=True),
                          pd.DataFrame(pca.transform(E_train), columns=emb_cols)], axis=1)
    Xe_test = pd.concat([test_df[FEATURE_COLUMNS].reset_index(drop=True),
                         pd.DataFrame(pca.transform(E_test), columns=emb_cols)], axis=1)
    y_train, y_test = train_df["label_trustworthy"], test_df["label_trustworthy"]

    both = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=cfg.seed).fit(Xe_train, y_train)
    emb_only = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=cfg.seed).fit(Xe_train[emb_cols], y_train)
    display(pd.DataFrame({
        "engineered features (calibrated)": binary_metrics(y_test, results["p_test"]),
        "embeddings only": binary_metrics(y_test, emb_only.predict_proba(Xe_test[emb_cols])[:, 1]),
        "engineered + embeddings": binary_metrics(y_test, both.predict_proba(Xe_test)[:, 1]),
    }).T.round(3))
else:
    print("USE_CODE_EMBEDDINGS = False: skipping the pretrained-encoder comparison.")

## 17. Methodology note for your write-up

> We use MBPP, a function-level Python benchmark with crowd-sourced problem statements and unit
> tests, as the task source. For each problem, one test is shown to the generating model
> (*visible*) and the remaining tests, including MBPP's challenge tests, are held back (*hidden*).
> Candidate solutions are sampled from several models and temperatures to obtain a realistic mix of
> correct and incorrect code. Each candidate is labelled **trustworthy** if it passes every hidden
> test in a sandboxed subprocess and has no medium- or high-severity Bandit finding; no AI model is
> asked to judge the code. A stratified sample was reviewed by the team to check that this label
> agrees with human judgments of trustworthiness (Cohen's kappa).
>
> The classifier only sees information available before the hidden tests are run: AST structure and
> complexity, lint and security findings, whether the visible test passes, agreement of the
> candidate's behaviour with sibling samples on probe inputs derived from the visible test, and the
> generating model's token log-probabilities. Data are split by problem into training (60%),
> calibration (20%) and test (20%) sets. Models are compared with grouped cross-validation against a
> majority-class and a visible-test-only baseline, and the selected model's probabilities are
> calibrated. We report accuracy, precision, recall, F1, ROC AUC, PR AUC and Brier score on the
> test set. A three-way recommendation (APPROVED / REJECTED / NEEDS HUMAN REVIEW) is produced from
> thresholds chosen on the calibration set to reach a target precision, and alternatively from
> split-conformal prediction; we report the share of candidates decided automatically and the error
> rate among those decisions (risk–coverage).

**Limitations to disclose**
- MBPP problems are short, isolated functions; results may not transfer to repository-level code.
- MBPP's hidden tests are few (usually two per problem), so some "passing" code is still wrong.
- MBPP is old enough to be in many models' training data; a LiveCodeBench slice (problems published
  after the models' training cut-off) would test for contamination.
- Bandit rarely fires on MBPP-style code, so the security part of the label is mostly inactive.
- Log-probabilities are computed after temperature / top-p adjustments and aren't available for
  Anthropic models; they are imputed.
- Mutant augmentation (if enabled) produces bugs with a different "style" from natural model errors;
  mutants are therefore used for training only, never for calibration or testing.
- If `HUMAN_ADDED_FEEDBACK = True` or `SMOKE_TEST = True` produced any reported number, say so:
  those runs show the pipeline works, not a finding about trustworthiness.